In [7]:
import plotly.express as px
from utils.data import *

start = dt.datetime(year=2000, month=1, day=1)
end = dt.datetime(year=2021, month=1, day=1)
all_stocks = ["NVDA", "AAPL", 'GOOG', "AVGO", "WMT", "JPM", "LLY", "XOM", "KO"]

n = len(all_stocks)

pce = load_pce()
effr = load_ffr()

a = (0.0001, 0)
r = (1+a[1])**(1/4)-1
mu = generate_mu(all_stocks, start, end)
Sigma = generate_Sigma(all_stocks, start, end)

new_sigma, sep_Sigma = add_covariates_to_covar(Sigma, all_stocks, [pce, effr], start, end)
new_mu, sep_mu = add_covariates_to_mu(mu, [pce, effr])

mu, Sigma = conditional_moments(sep_mu, sep_Sigma, a)

all_stocks = ["NVDA", "AAPL", 'GOOG', "AVGO", "WMT", "JPM", "LLY", "XOM", "KO"]

dfs = []
weights = [0.0, 0.0, 0.0, 0.6736609114412633, 0.3263390885587367, 0.0, 0.0, 0.0, 0.0]
deltas = [0.5, 0.5, 0.4153125732954087, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
print(weights)
w = np.array([w if w > 0.001 else 0 for w in weights ])
w /= w.sum()
 

[0.0, 0.0, 0.0, 0.6736609114412633, 0.3263390885587367, 0.0, 0.0, 0.0, 0.0]


In [10]:
import py_vollib.ref_python.black_scholes.greeks.analytical as pyv
import py_vollib.ref_python.black as black
import datetime as dt
import yfinance as yf
from dateutil.relativedelta import relativedelta

def getGreeks(date, expiry, stockPrice, r, sigma, strike):
    flag = 'p'
    dateDt = dt.datetime.strptime(date, "%Y-%m-%d")
    expiryDt = dt.datetime.strptime(expiry, "%Y-%m-%d")

    t = (expiryDt-dateDt).days/365.25

    delta = pyv.delta(flag, stockPrice, strike, t, r, sigma)
    gamma = pyv.gamma(flag, stockPrice, strike, t, r, sigma)
    theta = pyv.theta(flag, stockPrice, strike, t, r, sigma)
    vega = pyv.vega(flag, stockPrice, strike, t, r, sigma)
    rho  = pyv.rho(flag, stockPrice, strike, t, r, sigma)
    return delta, gamma, vega, theta, rho

def get_delta(date, expiry, stockPrice, r, sigma, strike):
    flag = 'p'
    dateDt = dt.datetime.strptime(date, "%Y-%m-%d")
    expiryDt = dt.datetime.strptime(expiry, "%Y-%m-%d")

    t = (expiryDt-dateDt).days/365.25
    print(t)
    return pyv.delta(flag, stockPrice, strike, t, r, sigma)

def quarter_start(start):
    return yf.Ticker("AAPL").history(start=start, interval="1d").index[0]

def quarter_end(start):
    end = start+relativedelta(months=3)
    return yf.Ticker("AAPL").history(end=end, interval="1d").index[-1]

def get_strike_from_delta(target_delta, initial_guess, qs, qe, r, stock_price, sigma):
    x1, x2 = (0, initial_guess)
    finished = False
    print(f'target delta: {target_delta}')
    
    
    tol = 1e-3
    while not finished:
        #print(f"delta at x2 (ATM-ish): {-get_delta(qs, qe, stock_price, r, sigma, stock_price):.4f}")
        #print(f"delta at x1 (near 0):  {-get_delta(qs, qe, stock_price, r, sigma, 1.0):.4f}")
        midpoint = (x1+x2)/2
        print(f'Midpoint: {x1/2+x2/2}')
        
        delta = -get_delta(qs, qe, stock_price, r, sigma, midpoint)
        print(f"delta at midpoint:  {delta:.4f}")
        if target_delta-delta > 0:
            x1 = midpoint
        else:
            x2 = midpoint
    
        if abs(target_delta - delta) < tol: finished= True

    
    return x1/2+x2/2

In [20]:
dfs = []
num_quarters = 1

for i, ticker in enumerate(all_stocks):
    if not weights[i]: continue
    

    df = pd.DataFrame(columns =['Open', 'price_with_put'])
    for q in range(num_quarters):
        period_start = (end + relativedelta(months=3*q))
        period_end = (end + relativedelta(months=3*(q+1)))

        hist = yf.Ticker(ticker).history(start=period_start, end=period_end, interval="1d")[['Open']]

        
        hist["price_with_put"] = hist.Open - hist.Open.iloc[0] + previous_value if q else hist.Open

        
        strike = get_strike_from_delta(deltas[i], hist.Open.iloc[0]*2, period_start.strftime("%Y-%m-%d"), period_end.strftime("%Y-%m-%d"), r, hist.Open.iloc[0], np.sqrt(Sigma[i,i])*2)
        
        num_days = len(hist.index)

        put_value = np.array([black.black_put(hist.Open.iloc[d], strike, (period_end-hist.reset_index().Date.apply(lambda x: x.replace(tzinfo=None)).iloc[d]).days/365.25, r, np.sqrt(Sigma[i,i])*2) for d in range(num_days)])
        #hist["put_value"] = put_value
        
        put_value -= put_value[0]
        #print(put_value, strike, hist.Open.iloc[0])
        hist.price_with_put += put_value
        
        previous_value = hist.price_with_put.iloc[-1]
        df = hist if not len(df.index) else pd.concat([df, hist])

    #df.Open *= w[i]/df.Open.iloc[0]
    #df.price_with_put *= w[i]/df.price_with_put.iloc[0]
    dfs.append(df)

init_df = dfs[0]

for i, d in enumerate(dfs):
    if i:
        init_df = init_df.add(d, fill_value=0)

target delta: 0.1
Midpoint: 39.43971522749755
0.2464065708418891
delta at midpoint:  0.4744
Midpoint: 19.719857613748776
0.2464065708418891
delta at midpoint:  0.0000
Midpoint: 29.579786420623165
0.2464065708418891
delta at midpoint:  0.0105
Midpoint: 34.50975082406036
0.2464065708418891
delta at midpoint:  0.1344
Midpoint: 32.04476862234176
0.2464065708418891
delta at midpoint:  0.0461
Midpoint: 33.27725972320106
0.2464065708418891
delta at midpoint:  0.0824
Midpoint: 33.89350527363071
0.2464065708418891
delta at midpoint:  0.1063
Midpoint: 33.58538249841588
0.2464065708418891
delta at midpoint:  0.0939
Midpoint: 33.739443886023295
0.2464065708418891
delta at midpoint:  0.1000
target delta: 0.1
Midpoint: 44.847978468236256
0.2464065708418891
delta at midpoint:  0.4840
Midpoint: 22.423989234118128
0.2464065708418891
delta at midpoint:  -0.0000
Midpoint: 33.635983851177194
0.2464065708418891
delta at midpoint:  0.0001
Midpoint: 39.241981159706725
0.2464065708418891
delta at midpoint:  0

In [21]:
px.line(init_df)